# Расчет метрик

In [576]:
from typing import List, Dict, Any

import pandas as pd
from pandas import DataFrame
import numpy as np

In [577]:
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.2f}'.format)

## ABC-анализ

Берутся все продажи, группируются по товарам, считается на какую сумму продано каждого товара за некоторый период времени или сколько штук каждого товара продалось за период времени. Затем считается вклад каждого товара (суммы или количества) в "общий котел". Затем, сортируется, считается сумма накопительным итогом, после этого выполняется классификация по правилу Парето: выбираем товары, которые накопительным итогом приносят 80% прибыли - группа A, с 80% до 95 - группа B и последние 5% - группа C.

In [578]:
df: pd.DataFrame = pd.read_csv(
    "data/data/data.csv",
    delimiter=",",
    header=0,
    encoding='1251'
)

df

,DR_Dat,DR_Tim,DR_NChk,DR_NDoc,DR_Apt,DR_Kkm,DR_TDoc,DR_TPay,DR_CDrugs,DR_NDrugs,DR_Suppl,DR_Prod,DR_Kol,DR_CZak,DR_CRoz,DR_SDisc,DR_CDisc,DR_BCDisc,DR_TabEmpl,DR_VZak,DR_Pos
0,2022-08-11,10:15:35,2173,2004598,2,22577,Розничная реализация,18,45399,ЦИПРОЛЕТ 3МГ/МЛ. 5МЛ. №1 ГЛ.КАПЛИ ФЛ./КАП. /Д-...,Катрен г.Химки,Д-р Редди с Лабораторис Лтд / Dr.REDDY's,1.00,41.08,51.00,12.00,925.00,200000000492.00,205,1,1.00
1,2022-08-11,10:27:46,2174,2004598,2,22577,Розничная реализация,15,261519,ПЕРЕКИСЬ ВОДОРОДА 3% 100МЛ. №40 Р-Р ФЛ.,Катрен г.Химки,ФЛОРА КАВКАЗА ОАО,1.00,18.61,31.00,3.00,9.00,200010010204.00,205,1,1.00
2,2022-08-11,10:27:46,2174,2004598,2,22577,Розничная реализация,15,460864,СОФЬЯ ГЕЛЬ Д/НОГ ВЕНОТОНИЗ. ТРОКСЕРУТИН ФОРТЕ ...,Катрен г.Химки,КОРОЛЕВФАРМ ООО,1.00,132.69,209.00,20.00,9.00,200010010204.00,205,1,2.00
3,2022-08-11,10:27:46,2174,2004598,2,22577,Розничная реализация,15,172823,СОФЬЯ ГХК КРЕМ Д/ТЕЛА ХОНДРОИТИН+ГЛЮКОЗАМИН 12...,Катрен г.Химки,КОРОЛЕВФАРМ ООО,1.00,133.65,210.00,21.00,9.00,200010010204.00,205,1,3.00
4,2022-08-11,10:33:56,2175,2004598,2,22577,Розничная реализация,18,79056,ГАЛВУС 50МГ. №28 ТАБ. /НОВАРТИС/,Катрен г.Химки,Новартис Фарма АГ,1.00,709.95,787.00,49.00,925.00,200000000492.00,205,1,1.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4457,2022-08-12,21:40:17,5690,18002542,18,9907,Розничная реализация,18,463100,КЕТОРОЛ ЭКСПРЕСС 10МГ. №20 ТАБ. ДИСПЕРГ. /Д-Р ...,ГРАНД КАПИТАЛ СМОЛЕНСК ООО ФК,Д-р Редди с Лабораторис Лтд / Dr.REDDY's,1.00,47.88,75.00,0.00,NaN,NaN,48,1,1.00
4458,2022-08-12,21:40:59,5691,18002542,18,9907,Розничная реализация,18,112158,АНТИПОЛИЦАЙ ВАЙТ №6 ТАБ.,Протек,ИНАТ-ФАРМА,1.00,59.51,93.00,0.00,NaN,NaN,48,1,1.00
4459,2022-08-12,21:43:48,5692,18002542,18,9907,Розничная реализация,18,260990,СИЛДЕНАФИЛ-СЗ 50МГ. №10 ТАБ. П/П/О /СЕВЕРНАЯ З...,Авеста,Северная Звезда,1.00,297.74,396.00,0.00,NaN,NaN,48,1,1.00
4460,2022-08-12,21:44:46,5693,18002542,18,9907,Розничная реализация,18,41512,ТОБРАЗОН 5МЛ. ГЛ.КАПЛИ ФЛ.,Катрен г.Химки,Кадила Фармасьютикалз Лтд ( CADILA ),1.00,322.06,419.00,0.00,NaN,NaN,48,1,1.00


In [579]:
# Сгруппируем данные по количеству проданных штук
grouped_df = df.groupby("DR_NDrugs").agg({"DR_Kol": "sum"})
grouped_df

,DR_Kol
DR_NDrugs,
7 СЕМЬ ДНЕЙ МАСКА Д/ЛИЦА СУББОТА РОМАНТ. ПЕРЕД СВИДАНИЕМ КРАСН.АПЕЛЬС.+ПАПАЙЯ 28МЛ. №1 [7 DAYS VILEN,1.00
"9 МЕСЯЦЕВ ФОЛИЕВАЯ К-ТА 0,4МГ. №30 ТАБ. П/П/О /ВАЛЕНТА/",1.00
911-ГЕЛЬ-БАЛЬЗАМ Д/СУСТАВОВ ОКОПНИК 100МЛ. ТУБА,2.00
911-ГЕЛЬ-БАЛЬЗАМ Д/СУСТАВОВ САБЕЛЬНИК 100МЛ.,2.00
911-ГЕЛЬ-БАЛЬЗАМ Д/ТЕЛА БИШОФИТ П/БОЛИ В СУСТАВАХ И МЫШЦАХ 100МЛ.,1.00
...,...
Я САМАЯ ВАТНЫЕ ПАЛОЧКИ №100 СТАКАН,1.00
ЯНТАРНАЯ К-ТА 100МГ. №10 ТАБ. (БАД) /МАРБИОФАРМ/,2.00
ЯНТАРНАЯ К-ТА ПРЕМИУМ 100МГ. №20 ТАБ. (БАД) /БАРНАУЛЬСКИЙ ЗМП/,1.00


In [580]:
# Теперь поделим количество продаж каждого товара
# на общее количество проданных товаров
grouped_df = (grouped_df["DR_Kol"] / sum(grouped_df["DR_Kol"])).sort_values(ascending=False)
grouped_df

DR_NDrugs
ПАКЕТ                                                                         0.03
ЦЕФТРИАКСОН 1Г. №1 ПОР. Д/Р-РА Д/В/В,В/М ФЛ. /ЛЕККО/                          0.01
НАФТИЗИН 0,1% 15МЛ. НАЗАЛ.КАПЛИ ФЛ./КАП. /ЛЕККО/                              0.01
ЛЕЙКОПЛАСТЫРЬ БАКТЕР. 2,5Х7,2 №1 /ВЕРОФАРМ/                                   0.01
СНУП 0,1% 90МКГ/ДОЗА 15МЛ. НАЗАЛ.СПРЕЙ ФЛ. /ШТАДА/                            0.01
                                                                              ... 
НАТРИЯ ХЛОРИД 0,9% 400МЛ. №16 Р-Р Д/ИНФ. КОНТ. /МОСФАРМ/                      0.00
СФМ ШПРИЦ 2МЛ. 3-Х КОМП. 0,63Х32ММ 23G №100 [SFM]                             0.00
СФМ ШПРИЦ 50МЛ. 3-Х КОМП. 1,2X40ММ 18G №25 [SFM]                              0.00
ДЕРМАГРИП ХАЙ РИСК ПЕРЧАТКИ ЛАТ. СМОТР. Н/СТЕР. Н/ОПУДР Р.L №50 (25ПАР)       0.00
СФМ ПЕРЧАТКИ СМОТР. Н/СТЕР. ЛАТ. ТЕКСТУРИР. Н/ОПУДР. Р.L №100 (50ПАР) [SFM]   0.00
Name: DR_Kol, Length: 1876, dtype: float64

In [581]:
# Считаем сумму накопительным итогом

grouped_df = grouped_df.cumsum().reset_index()
grouped_df

,DR_NDrugs,DR_Kol
0,ПАКЕТ,0.03
1,"ЦЕФТРИАКСОН 1Г. №1 ПОР. Д/Р-РА Д/В/В,В/М ФЛ. /...",0.04
2,"НАФТИЗИН 0,1% 15МЛ. НАЗАЛ.КАПЛИ ФЛ./КАП. /ЛЕККО/",0.05
3,"ЛЕЙКОПЛАСТЫРЬ БАКТЕР. 2,5Х7,2 №1 /ВЕРОФАРМ/",0.06
4,"СНУП 0,1% 90МКГ/ДОЗА 15МЛ. НАЗАЛ.СПРЕЙ ФЛ. /ШТ...",0.07
...,...,...
1871,"НАТРИЯ ХЛОРИД 0,9% 400МЛ. №16 Р-Р Д/ИНФ. КОНТ....",1.00
1872,"СФМ ШПРИЦ 2МЛ. 3-Х КОМП. 0,63Х32ММ 23G №100 [SFM]",1.00
1873,"СФМ ШПРИЦ 50МЛ. 3-Х КОМП. 1,2X40ММ 18G №25 [SFM]",1.00
1874,ДЕРМАГРИП ХАЙ РИСК ПЕРЧАТКИ ЛАТ. СМОТР. Н/СТЕР...,1.00


In [582]:
grouped_df["abc"] = np.where(
    grouped_df["DR_Kol"] < 0.8, "A", 
    np.where(
        grouped_df["DR_Kol"] < 0.95, "B", 
        "C"
    )
)
grouped_df

,DR_NDrugs,DR_Kol,abc
0,ПАКЕТ,0.03,A
1,"ЦЕФТРИАКСОН 1Г. №1 ПОР. Д/Р-РА Д/В/В,В/М ФЛ. /...",0.04,A
2,"НАФТИЗИН 0,1% 15МЛ. НАЗАЛ.КАПЛИ ФЛ./КАП. /ЛЕККО/",0.05,A
3,"ЛЕЙКОПЛАСТЫРЬ БАКТЕР. 2,5Х7,2 №1 /ВЕРОФАРМ/",0.06,A
4,"СНУП 0,1% 90МКГ/ДОЗА 15МЛ. НАЗАЛ.СПРЕЙ ФЛ. /ШТ...",0.07,A
...,...,...,...
1871,"НАТРИЯ ХЛОРИД 0,9% 400МЛ. №16 Р-Р Д/ИНФ. КОНТ....",1.00,C
1872,"СФМ ШПРИЦ 2МЛ. 3-Х КОМП. 0,63Х32ММ 23G №100 [SFM]",1.00,C
1873,"СФМ ШПРИЦ 50МЛ. 3-Х КОМП. 1,2X40ММ 18G №25 [SFM]",1.00,C
1874,ДЕРМАГРИП ХАЙ РИСК ПЕРЧАТКИ ЛАТ. СМОТР. Н/СТЕР...,1.00,C


In [583]:
grouped_df[grouped_df["abc"] == "B"]

,DR_NDrugs,DR_Kol,abc
938,ВИТАМИН С 250МГ. №20 ШИП.ТАБ. /ХЕМОФАРМ/,0.80,B
939,ВИТАМИР ВИТАМИН К2 МК7 100МКГ. №30 ТАБ. /КВАДР...,0.80,B
940,ВИТАПРОСТ ФОРТЕ 20МГ. №10 СУПП. РЕКТ. /НИЖФАРМ/,0.80,B
941,"РИНОСТОП ЭКСТРА 0,05% 15МЛ. НАЗАЛ.СПРЕЙ ФЛ.",0.80,B
942,РИБОКСИН 2% 5МЛ. №10 Р-Р Д/В/В ВВЕД. АМП. /БИО...,0.80,B
...,...,...,...
1621,"ДЕКСАЛГИН 25МГ/МЛ. 2МЛ. №5 Р-Р Д/ИН. Д/В/М,В/В...",0.95,B
1622,"ДЕКСАМЕТАЗОН РЕНЕВАЛ 0,1% 10МЛ. №1 ГЛ.КАПЛИ ТЮ...",0.95,B
1623,ГУТТАЛАКС ЭКСПРЕСС 10МГ. №6 СУПП. РЕКТ. /САНОФИ/,0.95,B
1624,"ОПАТАНОЛ 0,1% 5МЛ. ГЛ.КАПЛИ ФЛ./КАП.",0.95,B


In [584]:
# Проведем многомерный ABC-анализ, но уже с другими параметрами
# Анализ будем проводить по каоличеству проданных товаром 
# и по сумме розничных продаж

df: pd.DataFrame = pd.read_csv(
    "data/data/data.csv",
    delimiter=",",
    header=0,
    encoding='1251'
)

df

,DR_Dat,DR_Tim,DR_NChk,DR_NDoc,DR_Apt,DR_Kkm,DR_TDoc,DR_TPay,DR_CDrugs,DR_NDrugs,DR_Suppl,DR_Prod,DR_Kol,DR_CZak,DR_CRoz,DR_SDisc,DR_CDisc,DR_BCDisc,DR_TabEmpl,DR_VZak,DR_Pos
0,2022-08-11,10:15:35,2173,2004598,2,22577,Розничная реализация,18,45399,ЦИПРОЛЕТ 3МГ/МЛ. 5МЛ. №1 ГЛ.КАПЛИ ФЛ./КАП. /Д-...,Катрен г.Химки,Д-р Редди с Лабораторис Лтд / Dr.REDDY's,1.00,41.08,51.00,12.00,925.00,200000000492.00,205,1,1.00
1,2022-08-11,10:27:46,2174,2004598,2,22577,Розничная реализация,15,261519,ПЕРЕКИСЬ ВОДОРОДА 3% 100МЛ. №40 Р-Р ФЛ.,Катрен г.Химки,ФЛОРА КАВКАЗА ОАО,1.00,18.61,31.00,3.00,9.00,200010010204.00,205,1,1.00
2,2022-08-11,10:27:46,2174,2004598,2,22577,Розничная реализация,15,460864,СОФЬЯ ГЕЛЬ Д/НОГ ВЕНОТОНИЗ. ТРОКСЕРУТИН ФОРТЕ ...,Катрен г.Химки,КОРОЛЕВФАРМ ООО,1.00,132.69,209.00,20.00,9.00,200010010204.00,205,1,2.00
3,2022-08-11,10:27:46,2174,2004598,2,22577,Розничная реализация,15,172823,СОФЬЯ ГХК КРЕМ Д/ТЕЛА ХОНДРОИТИН+ГЛЮКОЗАМИН 12...,Катрен г.Химки,КОРОЛЕВФАРМ ООО,1.00,133.65,210.00,21.00,9.00,200010010204.00,205,1,3.00
4,2022-08-11,10:33:56,2175,2004598,2,22577,Розничная реализация,18,79056,ГАЛВУС 50МГ. №28 ТАБ. /НОВАРТИС/,Катрен г.Химки,Новартис Фарма АГ,1.00,709.95,787.00,49.00,925.00,200000000492.00,205,1,1.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4457,2022-08-12,21:40:17,5690,18002542,18,9907,Розничная реализация,18,463100,КЕТОРОЛ ЭКСПРЕСС 10МГ. №20 ТАБ. ДИСПЕРГ. /Д-Р ...,ГРАНД КАПИТАЛ СМОЛЕНСК ООО ФК,Д-р Редди с Лабораторис Лтд / Dr.REDDY's,1.00,47.88,75.00,0.00,NaN,NaN,48,1,1.00
4458,2022-08-12,21:40:59,5691,18002542,18,9907,Розничная реализация,18,112158,АНТИПОЛИЦАЙ ВАЙТ №6 ТАБ.,Протек,ИНАТ-ФАРМА,1.00,59.51,93.00,0.00,NaN,NaN,48,1,1.00
4459,2022-08-12,21:43:48,5692,18002542,18,9907,Розничная реализация,18,260990,СИЛДЕНАФИЛ-СЗ 50МГ. №10 ТАБ. П/П/О /СЕВЕРНАЯ З...,Авеста,Северная Звезда,1.00,297.74,396.00,0.00,NaN,NaN,48,1,1.00
4460,2022-08-12,21:44:46,5693,18002542,18,9907,Розничная реализация,18,41512,ТОБРАЗОН 5МЛ. ГЛ.КАПЛИ ФЛ.,Катрен г.Химки,Кадила Фармасьютикалз Лтд ( CADILA ),1.00,322.06,419.00,0.00,NaN,NaN,48,1,1.00


In [585]:
grouped_data: pd.DataFrame = df.groupby("DR_NDrugs").agg({
    "DR_Kol": "sum", 
    "DR_CRoz": "sum"
})

grouped_data

,DR_Kol,DR_CRoz
DR_NDrugs,,
7 СЕМЬ ДНЕЙ МАСКА Д/ЛИЦА СУББОТА РОМАНТ. ПЕРЕД СВИДАНИЕМ КРАСН.АПЕЛЬС.+ПАПАЙЯ 28МЛ. №1 [7 DAYS VILEN,1.00,93.00
"9 МЕСЯЦЕВ ФОЛИЕВАЯ К-ТА 0,4МГ. №30 ТАБ. П/П/О /ВАЛЕНТА/",1.00,144.00
911-ГЕЛЬ-БАЛЬЗАМ Д/СУСТАВОВ ОКОПНИК 100МЛ. ТУБА,2.00,216.00
911-ГЕЛЬ-БАЛЬЗАМ Д/СУСТАВОВ САБЕЛЬНИК 100МЛ.,2.00,208.00
911-ГЕЛЬ-БАЛЬЗАМ Д/ТЕЛА БИШОФИТ П/БОЛИ В СУСТАВАХ И МЫШЦАХ 100МЛ.,1.00,100.00
...,...,...
Я САМАЯ ВАТНЫЕ ПАЛОЧКИ №100 СТАКАН,1.00,65.00
ЯНТАРНАЯ К-ТА 100МГ. №10 ТАБ. (БАД) /МАРБИОФАРМ/,2.00,27.00
ЯНТАРНАЯ К-ТА ПРЕМИУМ 100МГ. №20 ТАБ. (БАД) /БАРНАУЛЬСКИЙ ЗМП/,1.00,52.00


In [586]:
# Сначала делаем анализ по количеству

grouped_data["rel_kol"] = (grouped_data["DR_Kol"] / sum(grouped_data["DR_Kol"]))
grouped_data

,DR_Kol,DR_CRoz,rel_kol
DR_NDrugs,,,
7 СЕМЬ ДНЕЙ МАСКА Д/ЛИЦА СУББОТА РОМАНТ. ПЕРЕД СВИДАНИЕМ КРАСН.АПЕЛЬС.+ПАПАЙЯ 28МЛ. №1 [7 DAYS VILEN,1.00,93.00,0.00
"9 МЕСЯЦЕВ ФОЛИЕВАЯ К-ТА 0,4МГ. №30 ТАБ. П/П/О /ВАЛЕНТА/",1.00,144.00,0.00
911-ГЕЛЬ-БАЛЬЗАМ Д/СУСТАВОВ ОКОПНИК 100МЛ. ТУБА,2.00,216.00,0.00
911-ГЕЛЬ-БАЛЬЗАМ Д/СУСТАВОВ САБЕЛЬНИК 100МЛ.,2.00,208.00,0.00
911-ГЕЛЬ-БАЛЬЗАМ Д/ТЕЛА БИШОФИТ П/БОЛИ В СУСТАВАХ И МЫШЦАХ 100МЛ.,1.00,100.00,0.00
...,...,...,...
Я САМАЯ ВАТНЫЕ ПАЛОЧКИ №100 СТАКАН,1.00,65.00,0.00
ЯНТАРНАЯ К-ТА 100МГ. №10 ТАБ. (БАД) /МАРБИОФАРМ/,2.00,27.00,0.00
ЯНТАРНАЯ К-ТА ПРЕМИУМ 100МГ. №20 ТАБ. (БАД) /БАРНАУЛЬСКИЙ ЗМП/,1.00,52.00,0.00


In [587]:
grouped_data = grouped_data.sort_values("rel_kol", ascending=False)

In [588]:
grouped_data["cumsum_kol"] = grouped_data["rel_kol"].cumsum()
grouped_data

,DR_Kol,DR_CRoz,rel_kol,cumsum_kol
DR_NDrugs,,,,
ПАКЕТ,126.00,248.00,0.03,0.03
"ЦЕФТРИАКСОН 1Г. №1 ПОР. Д/Р-РА Д/В/В,В/М ФЛ. /ЛЕККО/",60.00,2280.00,0.01,0.04
"НАФТИЗИН 0,1% 15МЛ. НАЗАЛ.КАПЛИ ФЛ./КАП. /ЛЕККО/",48.00,552.00,0.01,0.05
"ЛЕЙКОПЛАСТЫРЬ БАКТЕР. 2,5Х7,2 №1 /ВЕРОФАРМ/",40.00,28.00,0.01,0.06
"СНУП 0,1% 90МКГ/ДОЗА 15МЛ. НАЗАЛ.СПРЕЙ ФЛ. /ШТАДА/",35.00,4215.00,0.01,0.07
...,...,...,...,...
"НАТРИЯ ХЛОРИД 0,9% 400МЛ. №16 Р-Р Д/ИНФ. КОНТ. /МОСФАРМ/",0.06,669.00,0.00,1.00
"СФМ ШПРИЦ 2МЛ. 3-Х КОМП. 0,63Х32ММ 23G №100 [SFM]",0.05,918.00,0.00,1.00
"СФМ ШПРИЦ 50МЛ. 3-Х КОМП. 1,2X40ММ 18G №25 [SFM]",0.04,1324.00,0.00,1.00


In [589]:
# По признаку количества проданных товаров
grouped_data["abc_kol"] = np.where(
    grouped_data["cumsum_kol"] < 0.8, "A", 
    np.where(
        grouped_data["cumsum_kol"] < 0.95, "B", 
        "C"
    )
)
grouped_data

,DR_Kol,DR_CRoz,rel_kol,cumsum_kol,abc_kol
DR_NDrugs,,,,,
ПАКЕТ,126.00,248.00,0.03,0.03,A
"ЦЕФТРИАКСОН 1Г. №1 ПОР. Д/Р-РА Д/В/В,В/М ФЛ. /ЛЕККО/",60.00,2280.00,0.01,0.04,A
"НАФТИЗИН 0,1% 15МЛ. НАЗАЛ.КАПЛИ ФЛ./КАП. /ЛЕККО/",48.00,552.00,0.01,0.05,A
"ЛЕЙКОПЛАСТЫРЬ БАКТЕР. 2,5Х7,2 №1 /ВЕРОФАРМ/",40.00,28.00,0.01,0.06,A
"СНУП 0,1% 90МКГ/ДОЗА 15МЛ. НАЗАЛ.СПРЕЙ ФЛ. /ШТАДА/",35.00,4215.00,0.01,0.07,A
...,...,...,...,...,...
"НАТРИЯ ХЛОРИД 0,9% 400МЛ. №16 Р-Р Д/ИНФ. КОНТ. /МОСФАРМ/",0.06,669.00,0.00,1.00,C
"СФМ ШПРИЦ 2МЛ. 3-Х КОМП. 0,63Х32ММ 23G №100 [SFM]",0.05,918.00,0.00,1.00,C
"СФМ ШПРИЦ 50МЛ. 3-Х КОМП. 1,2X40ММ 18G №25 [SFM]",0.04,1324.00,0.00,1.00,C


In [590]:
# Далее, посчитаем то же самое, только для суммы розничных продаж

grouped_data["rel_roz"] = (grouped_data["DR_CRoz"] / sum(grouped_data["DR_CRoz"]))
grouped_data = grouped_data.sort_values("rel_roz", ascending=False)

grouped_data["cumsum_roz"] = grouped_data["rel_roz"].cumsum()

grouped_data["abc_roz"] = np.where(
    grouped_data["cumsum_roz"] < 0.8, "A", 
    np.where(
        grouped_data["cumsum_roz"] < 0.95, "B", 
        "C"
    )
)
grouped_data



,DR_Kol,DR_CRoz,rel_kol,cumsum_kol,abc_kol,rel_roz,cumsum_roz,abc_roz
DR_NDrugs,,,,,,,,
"ТЕРАФЛЮ ЛИМОН ОТ ГРИППА И ПРОСТУДЫ 22,1Г. №14 ПОР. Д/Р-РА Д/ПРИЕМА ВНУТРЬ ПАК.",7.00,15638.00,0.00,0.33,A,0.01,0.01,A
НИМЕСИЛ 100МГ. 2Г. №30 ГРАН. Д/СУСП. Д/ПРИЕМА ВНУТРЬ ПАК. /ГУИДОТТИ/МЕНАРИНИ/,2.80,14445.00,0.00,0.61,A,0.01,0.02,A
ИНГАВИРИН 90МГ. №10 КАПС. /ВАЛЕНТА/,18.00,13750.00,0.00,0.19,A,0.01,0.03,A
ХАРТМАНН БРАНОЛИНД H ПОВЯЗКА СТЕР. 10Х20СМ. №30 ПЕРУАН.БАЛЬЗАМ /АРТ.4923462/ [BRANOLIND],0.17,12908.00,0.00,1.00,C,0.01,0.04,A
"ЭЛИКВИС 2,5МГ. №60 ТАБ. П/П/О /ПФАЙЗЕР/БРИСТОЛ-МАЙЕРС/",5.00,12731.00,0.00,0.41,A,0.01,0.05,A
...,...,...,...,...,...,...,...,...
АСКОРБИНОВАЯ К-ТА 25МГ. АПЕЛЬСИН №10 ТАБ. КРУТКА САХ. /АСКОПРОМ/,1.00,10.00,0.00,0.87,B,0.00,1.00,C
БАХИЛЫ №10 (5ПАР),1.00,10.00,0.00,0.86,B,0.00,1.00,C
АСКОРБИНОВАЯ К-ТА 25МГ. №10 ТАБ. КРУТКА САХ. /АСКОПРОМ/,7.00,9.00,0.00,0.33,A,0.00,1.00,C


In [591]:
grouped_data[["DR_CRoz", "DR_Kol", "abc_kol", "abc_roz"]].reset_index().sort_values("DR_CRoz", ascending=False)

,DR_NDrugs,DR_CRoz,DR_Kol,abc_kol,abc_roz
0,"ТЕРАФЛЮ ЛИМОН ОТ ГРИППА И ПРОСТУДЫ 22,1Г. №14 ...",15638.00,7.00,A,A
1,НИМЕСИЛ 100МГ. 2Г. №30 ГРАН. Д/СУСП. Д/ПРИЕМА ...,14445.00,2.80,A,A
2,ИНГАВИРИН 90МГ. №10 КАПС. /ВАЛЕНТА/,13750.00,18.00,A,A
3,ХАРТМАНН БРАНОЛИНД H ПОВЯЗКА СТЕР. 10Х20СМ. №3...,12908.00,0.17,C,A
4,"ЭЛИКВИС 2,5МГ. №60 ТАБ. П/П/О /ПФАЙЗЕР/БРИСТОЛ...",12731.00,5.00,A,A
...,...,...,...,...,...
1871,АСКОРБИНОВАЯ К-ТА 25МГ. АПЕЛЬСИН №10 ТАБ. КРУТ...,10.00,1.00,B,C
1872,БАХИЛЫ №10 (5ПАР),10.00,1.00,B,C
1873,АСКОРБИНОВАЯ К-ТА 25МГ. №10 ТАБ. КРУТКА САХ. /...,9.00,7.00,A,C
1874,КЛИНСА БАХИЛЫ СТАНДАРТ №2 (1ПАРА),7.00,2.00,A,C


In [592]:
# Напишем функцию для выполнения ABC-анализа

def do_abc(data: DataFrame, index: str) -> DataFrame:
    """
    Расчет параметров ABC-анализа

    Args:
        data (DataFrame): Подготовленный DataFrame, из которого убраны все ненужные поля
        index (str): Название поля, по которому будет производиться группировка

    Returns:
        DataFrame: результирующий DataFrame
    """
    
    # Берем список столбцов и убираем из него тот, 
    # по которому будем группировать данные
    column_names: List[str] = list(data.columns)
    column_names.remove(index)

    # Группируем данные
    grouped_data: DataFrame = data.groupby(index).agg({
        column: sum for column in column_names  # Используем Dict comprehension
    })

    # Считаем соответствие значений группе (A, B или C)
    for column in column_names:
        grouped_data[f"rel_{column}"] = grouped_data[column] / sum(grouped_data[column])
        grouped_data = grouped_data.sort_values(f"rel_{column}", ascending=False)
        grouped_data[f"cumsum_{column}"] = grouped_data[f"rel_{column}"].cumsum()

        grouped_data[f"abc_{column}"] = np.where(
            grouped_data[f"cumsum_{column}"] < 0.8, "A", 
            np.where(
                grouped_data[f"cumsum_{column}"] < 0.95, "B", 
                "C"
            )
        )

    result: DataFrame = grouped_data[column_names + [f"abc_{column}" for column in column_names]]

    return result.reset_index()


In [593]:
df: pd.DataFrame = pd.read_csv(
    "data/data/data.csv",
    delimiter=",",
    header=0,
    encoding='1251'
)

prepared_data = df[["DR_NDrugs", "DR_Kol", "DR_CRoz"]]

do_abc(prepared_data, "DR_NDrugs")

,DR_NDrugs,DR_Kol,DR_CRoz,abc_DR_Kol,abc_DR_CRoz
0,"ТЕРАФЛЮ ЛИМОН ОТ ГРИППА И ПРОСТУДЫ 22,1Г. №14 ...",7.00,15638.00,A,A
1,НИМЕСИЛ 100МГ. 2Г. №30 ГРАН. Д/СУСП. Д/ПРИЕМА ...,2.80,14445.00,A,A
2,ИНГАВИРИН 90МГ. №10 КАПС. /ВАЛЕНТА/,18.00,13750.00,A,A
3,ХАРТМАНН БРАНОЛИНД H ПОВЯЗКА СТЕР. 10Х20СМ. №3...,0.17,12908.00,C,A
4,"ЭЛИКВИС 2,5МГ. №60 ТАБ. П/П/О /ПФАЙЗЕР/БРИСТОЛ...",5.00,12731.00,A,A
...,...,...,...,...,...
1871,АСКОРБИНОВАЯ К-ТА 25МГ. АПЕЛЬСИН №10 ТАБ. КРУТ...,1.00,10.00,B,C
1872,БАХИЛЫ №10 (5ПАР),1.00,10.00,B,C
1873,АСКОРБИНОВАЯ К-ТА 25МГ. №10 ТАБ. КРУТКА САХ. /...,7.00,9.00,A,C
1874,КЛИНСА БАХИЛЫ СТАНДАРТ №2 (1ПАРА),2.00,7.00,A,C


## Динамика продаж

In [594]:
# Продажи за 5 дней
df: pd.DataFrame = pd.read_csv(
    "data/data/data1.csv",
    delimiter=",",
    header=0,
    encoding='1251'
)

df

,DR_Dat,DR_Tim,DR_NChk,DR_NDoc,DR_Apt,DR_Kkm,DR_TDoc,DR_TPay,DR_CDrugs,DR_NDrugs,DR_Suppl,DR_Prod,DR_Kol,DR_CZak,DR_CRoz,DR_SDisc,DR_CDisc,DR_BCDisc,DR_TabEmpl,DR_VZak,DR_Pos
0,2022-08-01,08:06:18,1272,13002561,13,22589,Розничная реализация,18,144734,ГАСТАЛ №12 ТАБ. Д/РАСС.,Пульс,TEVA Pharvaceutical Industries Ltd,1.00,196.71,270.00,0.00,NaN,NaN,29,1,1.00
1,2022-08-01,08:38:53,1273,13002561,13,22589,Розничная реализация,15,69661,"ТОБРОПТ 0,3% 5МЛ. №1 ГЛ.КАПЛИ ФЛ./КАП. /РОМФАРМ/",Пульс,РОМФАРМ КОМПАНИ ( ROMPHARM ),1.00,106.21,127.00,6.00,9.00,200010004357.00,29,1,1.00
2,2022-08-01,08:55:38,1274,13002561,13,22589,Розничная реализация,18,190635,ЭЛИКВИС 5МГ. №60 ТАБ. П/П/О /ПФАЙЗЕР/БРИСТОЛ-М...,ГРАНД КАПИТАЛ СМОЛЕНСК ООО ФК,Пфайзер,1.00,2320.99,2563.00,76.00,9.00,200010018491.00,29,1,1.00
3,2022-08-01,09:00:40,1275,13002561,13,22589,Розничная реализация,18,276370,АРБИДОЛ МАКСИМУМ 200МГ. №10 КАПС. /ОТИСИФАРМ/Ф...,Пульс,ОТИСИФАРМ ПАО,1.00,445.39,541.00,0.00,NaN,NaN,29,1,1.00
4,2022-08-01,09:04:05,1276,13002561,13,22589,Розничная реализация,15,2303,"ЭНАМ 2,5МГ. №20 ТАБ. /Д-Р РЕДДИ/",Протек,Д-р Редди с Лабораторис Лтд / Dr.REDDY's,1.00,18.04,22.00,1.00,9.00,200010000734.00,29,1,5.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1443,2022-08-05,19:30:09,1899,13002616,13,22589,Розничная реализация,15,317993,ПАНКРЕАТИН 25ЕД №60 ТАБ.КШ/РАСТВ. П/О /АВЕКСИМА/,ГРАНД КАПИТАЛ СМОЛЕНСК ООО ФК,Авексима ОАО,1.00,61.22,76.00,0.00,NaN,NaN,50,1,1.00
1444,2022-08-05,19:47:18,1900,13002616,13,22589,Розничная реализация,18,431,ДИКЛОФЕНАК 25МГ/МЛ. 3МЛ. №5 Р-Р Д/В/М ВВЕД. АМ...,ГРАНД КАПИТАЛ СМОЛЕНСК ООО ФК,Хемофарм А.Д. (HEMOFARM ),1.00,41.83,51.00,0.00,NaN,NaN,50,1,1.00
1445,2022-08-05,19:47:18,1900,13002616,13,22589,Розничная реализация,18,4139,СИРДАЛУД 4МГ. №30 ТАБ. /НОВАРТИС/,Фармкомплект ООО,Новартис Фарма АГ,1.00,266.99,322.00,0.00,NaN,NaN,50,1,2.00
1446,2022-08-05,19:49:47,1902,13002616,13,22589,Розничная реализация,15,259301,МЕРИДИАН МОЧЕПРИЕМНИК ДЕТСКИЙ PD2200 200МЛ. [M...,Здравсервис,МЕРИДИАН / MERIDIAN,4.00,7.26,12.00,4.00,9.00,200010016190.00,50,1,2.00


In [595]:
df["DR_Dat"].unique()

<StringArray>
['2022-08-01', '2022-08-02', '2022-08-03', '2022-08-04', '2022-08-05']
Length: 5, dtype: str

In [596]:
df: pd.DataFrame = pd.read_csv(
    "data/data/data1.csv",
    delimiter=",",
    header=0,
    encoding='1251'
)

revenue: DataFrame = df.groupby('DR_Dat')[['DR_Kol', 'DR_CRoz', 'DR_SDisc']].apply(lambda x: pd.Series({'revenue': sum(x['DR_Kol']*x['DR_CRoz'] - x['DR_SDisc'])})).reset_index()
revenue

,DR_Dat,revenue
0,2022-08-01,84681.52
1,2022-08-02,71389.37
2,2022-08-03,78050.82
3,2022-08-04,59187.36
4,2022-08-05,56458.81


In [597]:
# Окно (window) - это параметр, который говорит, в какому количеству
# значений применить аггрегирующую функцию. Так, window=2 означает, что
# нужно взять  текущее и предыдущее значение и применить к ним аггрегацию

df = df.groupby('DR_Dat')[['DR_Kol', 'DR_CRoz', 'DR_SDisc']].apply(lambda x: sum(x['DR_Kol']*x['DR_CRoz'] - x['DR_SDisc'])).reset_index()
df

,DR_Dat,0
0,2022-08-01,84681.52
1,2022-08-02,71389.37
2,2022-08-03,78050.82
3,2022-08-04,59187.36
4,2022-08-05,56458.81


In [598]:
df.columns = ['DR_Dat', 'revenue']
df

,DR_Dat,revenue
0,2022-08-01,84681.52
1,2022-08-02,71389.37
2,2022-08-03,78050.82
3,2022-08-04,59187.36
4,2022-08-05,56458.81


### Есть еще вариант

In [599]:
df['revenue_d'] = df['revenue'].rolling(2).apply(lambda x: (x.iloc[1] - x.iloc[0])/x.iloc[0])
df

,DR_Dat,revenue,revenue_d
0,2022-08-01,84681.52,NaN
1,2022-08-02,71389.37,-0.16
2,2022-08-03,78050.82,0.09
3,2022-08-04,59187.36,-0.24
4,2022-08-05,56458.81,-0.05


In [600]:
df['revenue_shifted'] = df['revenue'].shift(1)
df

,DR_Dat,revenue,revenue_d,revenue_shifted
0,2022-08-01,84681.52,NaN,NaN
1,2022-08-02,71389.37,-0.16,84681.52
2,2022-08-03,78050.82,0.09,71389.37
3,2022-08-04,59187.36,-0.24,78050.82
4,2022-08-05,56458.81,-0.05,59187.36


In [601]:
df["revenue_d2"] = (df["revenue"] - df["revenue_shifted"]) / df["revenue_shifted"]
df

,DR_Dat,revenue,revenue_d,revenue_shifted,revenue_d2
0,2022-08-01,84681.52,NaN,NaN,NaN
1,2022-08-02,71389.37,-0.16,84681.52,-0.16
2,2022-08-03,78050.82,0.09,71389.37,0.09
3,2022-08-04,59187.36,-0.24,78050.82,-0.24
4,2022-08-05,56458.81,-0.05,59187.36,-0.05


### И третий вариант

In [602]:
df["revenue_d3"] = df["revenue"].pct_change(1)
df

,DR_Dat,revenue,revenue_d,revenue_shifted,revenue_d2,revenue_d3
0,2022-08-01,84681.52,NaN,NaN,NaN,NaN
1,2022-08-02,71389.37,-0.16,84681.52,-0.16,-0.16
2,2022-08-03,78050.82,0.09,71389.37,0.09,0.09
3,2022-08-04,59187.36,-0.24,78050.82,-0.24,-0.24
4,2022-08-05,56458.81,-0.05,59187.36,-0.05,-0.05


## XYZ-анализ

Это метод классификации товаров или ресурсов по стабильности спроса и точности прогнозирования продаж, где группы X, Y и Z делятся на основе коэффициента вариации (0–10%, 10–25% и >25% соответственно)

Коэффициент вариации = стандартное отклонение / среднее значение

Основные группы XYZ-анализа:

- **X (стабильный спрос)**: Коэффициент вариации 0-10%. Спрос почти не меняется, его легко спрогнозировать. Поставки можно планировать с высокой точностью. Эти ресурсы нужно «держать под рукой»
- **Y (средний спрос)**: Коэффициент вариации 10-25%. Продажи имеют колебания (например, сезонность), но поддаются прогнозу. Этим ресурсам нужно найти баланс.
- **Z (нестабильный спрос**): Коэффициент вариации >25%. Спрос случайный, нерегулярный, прогнозированию почти не подлежит. Имеют крайне волатильный спрос. Эти ресурсы лучше сбывать по предзаказу.



In [603]:
df: pd.DataFrame = pd.read_csv(
    "data/data/data1.csv",
    delimiter=",",
    header=0,
    encoding='1251'
)

df.head()

,DR_Dat,DR_Tim,DR_NChk,DR_NDoc,DR_Apt,DR_Kkm,DR_TDoc,DR_TPay,DR_CDrugs,DR_NDrugs,DR_Suppl,DR_Prod,DR_Kol,DR_CZak,DR_CRoz,DR_SDisc,DR_CDisc,DR_BCDisc,DR_TabEmpl,DR_VZak,DR_Pos
0,2022-08-01,08:06:18,1272,13002561,13,22589,Розничная реализация,18,144734,ГАСТАЛ №12 ТАБ. Д/РАСС.,Пульс,TEVA Pharvaceutical Industries Ltd,1.00,196.71,270.00,0.00,NaN,NaN,29,1,1.00
1,2022-08-01,08:38:53,1273,13002561,13,22589,Розничная реализация,15,69661,"ТОБРОПТ 0,3% 5МЛ. №1 ГЛ.КАПЛИ ФЛ./КАП. /РОМФАРМ/",Пульс,РОМФАРМ КОМПАНИ ( ROMPHARM ),1.00,106.21,127.00,6.00,9.00,200010004357.00,29,1,1.00
2,2022-08-01,08:55:38,1274,13002561,13,22589,Розничная реализация,18,190635,ЭЛИКВИС 5МГ. №60 ТАБ. П/П/О /ПФАЙЗЕР/БРИСТОЛ-М...,ГРАНД КАПИТАЛ СМОЛЕНСК ООО ФК,Пфайзер,1.00,2320.99,2563.00,76.00,9.00,200010018491.00,29,1,1.00
3,2022-08-01,09:00:40,1275,13002561,13,22589,Розничная реализация,18,276370,АРБИДОЛ МАКСИМУМ 200МГ. №10 КАПС. /ОТИСИФАРМ/Ф...,Пульс,ОТИСИФАРМ ПАО,1.00,445.39,541.00,0.00,NaN,NaN,29,1,1.00
4,2022-08-01,09:04:05,1276,13002561,13,22589,Розничная реализация,15,2303,"ЭНАМ 2,5МГ. №20 ТАБ. /Д-Р РЕДДИ/",Протек,Д-р Редди с Лабораторис Лтд / Dr.REDDY's,1.00,18.04,22.00,1.00,9.00,200010000734.00,29,1,5.00


In [604]:
grouped_data = df.groupby(["DR_Dat", "DR_NDrugs"]).agg({
    "DR_Kol": "sum"
})
grouped_data

DR_Kol
DR_Dat     DR_NDrugs                                                 
2022-08-01 911-ВЕНОЛГОН ГЕЛЬ Д/НОГ ПРИ ТЯЖЕСТИ,БОЛИ,ОТЕКАХ...    1.00
           L-ТИРОКСИН 100МКГ. №100 ТАБ. /БЕРЛИН ХЕМИ/            1.00
           L-ТИРОКСИН 50МКГ. №50 ТАБ. /БЕРЛИН ХЕМИ/              1.00
           L-ТИРОКСИН 75МКГ. №100 ТАБ. /БЕРЛИН ХЕМИ/             2.00
           NF ВАТА ХИРУРГ. СТЕР. 100Г. /НЬЮФАРМ/                 1.00
...                                                               ...
2022-08-05 ЭВО ПАНТЕНОЛ ПОМАДА ГИГИЕН. 2,8Г. [EVO] /АВАНТА/      1.00
           ЭМОКСИПИН 10МГ/МЛ. 5МЛ. №1 ГЛ.КАПЛИ ФЛ. КРЫШ/КАП.     2.00
           ЭСВИЦИН СР-ВО П/ОБЛЫСЕНИЯ ЛОСЬОН-ТОНИК 250МЛ. Ф...    1.00
           ЭТОРИАКС 90МГ. №7 ТАБ. П/П/О                          1.00
           ЮНИДОКС СОЛЮТАБ 100МГ. №20 ТАБ.ДИСПЕРГ. /АСТЕЛЛАС/    1.00

[1089 rows x 1 columns]

In [605]:
# Нужно определить коэффициент вариации
h_data = grouped_data.reset_index().groupby('DR_NDrugs').agg({
    'DR_Dat': 'count'
}).reset_index()

h_data

,DR_NDrugs,DR_Dat
0,"911-ВЕНОЛГОН ГЕЛЬ Д/НОГ ПРИ ТЯЖЕСТИ,БОЛИ,ОТЕКА...",1
1,911-ГЕЛЬ-БАЛЬЗАМ Д/НОГ КОНСКИЙ КАШТАН 100МЛ. ТУБА,1
2,911-ГЕЛЬ-БАЛЬЗАМ Д/СУСТАВОВ РАЗОГР. ОКОПНИК+МУ...,1
3,L-ТИРОКСИН 100МКГ. №100 ТАБ. /БЕРЛИН ХЕМИ/,1
4,L-ТИРОКСИН 150МКГ. №100 ТАБ. /БЕРЛИН ХЕМИ/,1
...,...,...
798,ЭСВИЦИН СР-ВО П/ОБЛЫСЕНИЯ ЛОСЬОН-ТОНИК 250МЛ. ...,4
799,ЭСПУМИЗАН 40МГ. №25 КАПС. /БЕРЛИН-ХЕМИ/,1
800,ЭТОРИАКС 90МГ. №7 ТАБ. П/П/О,1
801,ЮНИДОКС СОЛЮТАБ 100МГ. №10 ТАБ.ДИСПЕРГ. /АСТЕЛ...,1


In [606]:
# Оставим только товары, которые продавались более одного дня
# Потому что, если товар продавался только один день, то 
# XYZ-анализ просто не имеет смысла

xyz_data = list(h_data[h_data['DR_Dat'] > 1]['DR_NDrugs'])
xyz_data

['L-ТИРОКСИН 50МКГ. №50 ТАБ. /БЕРЛИН ХЕМИ/',
 'L-ТИРОКСИН 75МКГ. №100 ТАБ. /БЕРЛИН ХЕМИ/',
 'АЗАРГА 10МГ/МЛ.+5МГ/МЛ. 5МЛ. №1 ГЛ.КАПЛИ ФЛ./КАП.',
 'АМОКСИЦИЛЛИН 500МГ. №16 КАПС. /ХЕМОФАРМ/',
 'АРБИДОЛ МАКСИМУМ 200МГ. №10 КАПС. /ОТИСИФАРМ/ФАРМСТАНДАРТ/',
 'АСЕПТИКА САЛФЕТКА СПИРТОВАЯ 60Х100 №20',
 'АСКОРБИНКА ЛУНТИК ВИТ.С КЛУБНИКА №10 ТАБ. (30Г.) /ФАРМ-ПРО/ТИГОДА-ФАРМ/',
 'АСКОРБИНКА ЛУНТИК ВИТ.С №10 ТАБ. (30Г.) /ФАРМ-ПРО/ТИГОДА-ФАРМ/',
 'АСКОРБИНОВАЯ К-ТА 25МГ. ВИШНЯ №10 ТАБ. КРУТКА САХ. /АСКОПРОМ/',
 'АСКОРБИНОВАЯ К-ТА 25МГ. ЯБЛОКО №10 ТАБ. КРУТКА САХ. /АСКОПРОМ/',
 'АСПАРКАМ №60 ТАБ. /ФАРМАПОЛ-ВОЛГА/',
 'АСПЕРА СУПЕРЧИСТОТЕЛ 3МЛ. ФЛ.',
 'АТЕНОЛОЛ РЕНЕВАЛ 50МГ. №30 ТАБ. /ОБНОВЛЕНИЕ/',
 'АТОПИК КРЕМ-СТИК УСПОКАИВАЮЩИЙ Д/ДЕТ. 0+ 4,9МЛ. ПЕНАЛ [ATOPIC]',
 'АТОРВАСТАТИН-СЗ 20МГ. №60 ТАБ. П/П/О /СЕВЕРНАЯ ЗВЕЗДА/',
 'АФОБАЗОЛ 10МГ. №60 ТАБ.',
 'АЦЕТИЛСАЛИЦИЛОВАЯ К-ТА 500МГ. №20 ТАБ. /ФАРМСТАНДАРТ/',
 'АЦИКЛОВИР-БЕЛУПО 5% 10Г. №1 КРЕМ Д/НАРУЖ.ПРИМ. ТУБА /БЕЛУПО/',
 'БЕТАГИСТИН-СЗ 16МГ. №60 ТАБ

In [607]:
grouped_data = grouped_data.reset_index()
grouped_data = grouped_data[grouped_data['DR_NDrugs'].isin(xyz_data)]
grouped_data = grouped_data.groupby('DR_NDrugs')[['DR_Kol']].apply(lambda x: x.std()/x.mean())
grouped_data

,DR_Kol
DR_NDrugs,
L-ТИРОКСИН 50МКГ. №50 ТАБ. /БЕРЛИН ХЕМИ/,0.00
L-ТИРОКСИН 75МКГ. №100 ТАБ. /БЕРЛИН ХЕМИ/,0.47
АЗАРГА 10МГ/МЛ.+5МГ/МЛ. 5МЛ. №1 ГЛ.КАПЛИ ФЛ./КАП.,0.00
АМОКСИЦИЛЛИН 500МГ. №16 КАПС. /ХЕМОФАРМ/,0.47
АРБИДОЛ МАКСИМУМ 200МГ. №10 КАПС. /ОТИСИФАРМ/ФАРМСТАНДАРТ/,0.43
...,...
"ЭВО ПАНТЕНОЛ ПОМАДА ГИГИЕН. 2,8Г. [EVO] /АВАНТА/",0.47
ЭЛИКВИС 5МГ. №60 ТАБ. П/П/О /ПФАЙЗЕР/БРИСТОЛ-МАЙЕРС/,0.00
ЭМОКСИПИН 10МГ/МЛ. 5МЛ. №1 ГЛ.КАПЛИ ФЛ. КРЫШ/КАП.,0.47


In [610]:
grouped_data['XYZ'] = np.where(grouped_data['DR_Kol'] < 0.1, 'X', np.where(grouped_data['DR_Kol'] < 0.25, 'Y', 'Z'))
grouped_data

,DR_Kol,XYZ
DR_NDrugs,,
L-ТИРОКСИН 50МКГ. №50 ТАБ. /БЕРЛИН ХЕМИ/,0.00,X
L-ТИРОКСИН 75МКГ. №100 ТАБ. /БЕРЛИН ХЕМИ/,0.47,Z
АЗАРГА 10МГ/МЛ.+5МГ/МЛ. 5МЛ. №1 ГЛ.КАПЛИ ФЛ./КАП.,0.00,X
АМОКСИЦИЛЛИН 500МГ. №16 КАПС. /ХЕМОФАРМ/,0.47,Z
АРБИДОЛ МАКСИМУМ 200МГ. №10 КАПС. /ОТИСИФАРМ/ФАРМСТАНДАРТ/,0.43,Z
...,...,...
"ЭВО ПАНТЕНОЛ ПОМАДА ГИГИЕН. 2,8Г. [EVO] /АВАНТА/",0.47,Z
ЭЛИКВИС 5МГ. №60 ТАБ. П/П/О /ПФАЙЗЕР/БРИСТОЛ-МАЙЕРС/,0.00,X
ЭМОКСИПИН 10МГ/МЛ. 5МЛ. №1 ГЛ.КАПЛИ ФЛ. КРЫШ/КАП.,0.47,Z
